# Neural posterior estimation

A network learns the posterior of the event parameters $\theta = (x_0, y_0, E)$ from simulated events: neural
posterior estimation (NPE). This notebook is a guided first look at the package `prob_reco_sbi.npe`: it simulates
pairs $\lbrace \theta_i, x_i \rbrace$, trains the chain of conditional spline flows on them, and draws the posterior
of a new event on the event display. The maths is in [docs/npe.md](../docs/npe.md) and the tasks are in
[tasks/stage2b_npe/](../tasks/stage2b_npe/README.md).

Run it in the environment of the [README](../README.md) with the kernel registered there. It takes well under a minute
on a CPU and writes nothing to disk.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

import prob_reco_sbi
from prob_reco_sbi.flash import generate, preset, sensor_positions, simulate
from prob_reco_sbi.npe import AutoregressiveFlow, fit, to_pairs

print("prob_reco_sbi", prob_reco_sbi.__version__, "| torch", torch.__version__)

In [ ]:
# The figure style of the course (the paper style of the NeuroMCT figures): white, no grid, ticks pointing in on
# all four sides with minor ticks, framed legends inside the axes; samples as step histograms, a density compared with
# them dashed in the same color.
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": ".15",
        "axes.labelcolor": ".15",
        "axes.linewidth": 1.25,
        "axes.grid": False,
        "axes.axisbelow": True,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.spines.right": True,
        "axes.spines.top": True,
        "text.color": ".15",
        "xtick.color": ".15",
        "ytick.color": ".15",
        "xtick.major.width": 1.25,
        "ytick.major.width": 1.25,
        "xtick.minor.width": 1.0,
        "ytick.minor.width": 1.0,
        "xtick.major.size": 6.0,
        "ytick.major.size": 6.0,
        "xtick.minor.size": 4.0,
        "ytick.minor.size": 4.0,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "Bitstream Vera Sans", "sans-serif"],
        "lines.linewidth": 1.5,
        "lines.markersize": 6.0,
        "patch.linewidth": 1.0,
        "patch.edgecolor": "w",
        "patch.force_edgecolor": True,
        "lines.solid_capstyle": "round",
        "legend.frameon": True,
        "image.cmap": "viridis",
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,
        "xtick.direction": "in",
        "xtick.top": True,
        "xtick.bottom": True,
        "xtick.labelsize": 12,
        "ytick.direction": "in",
        "ytick.right": True,
        "ytick.left": True,
        "ytick.labelsize": 12,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "pdf.fonttype": 42,
    }
)
# One color per meaning, the same in every figure of this notebook.
COLORS = {"samples": "indigo", "truth": "black", "barycenter": "darkorange", "train": "indigo", "val": "royalblue"}

## Pairs from the prior and the simulator

NPE learns from pairs: parameters $\theta_i$ drawn from the prior, and the observations $x_i$ that the simulator makes
for them. The prior is that of stage 1 (the vertex uniform in the disc, $E$ uniform in [0.5, 5] MeV), and the
observations are the 32 log charges and the 32 relative times of `network_input`. `to_pairs` turns simulated events
into the two tensors. The last 2000 pairs are kept for validation.

In [ ]:
config = preset("full")
start = time.perf_counter()
events = generate(22_000, config=config, seed=1)
theta, x = to_pairs(events, "charge+time")  # theta: x0, y0 [m], E [MeV]; x: the network input
print(f"{len(theta)} pairs in {time.perf_counter() - start:.0f} s: theta {tuple(theta.shape)}, x {tuple(x.shape)}")
print("first pair: theta =", theta[0].numpy().round(3), "| x[:4] =", x[0, :4].numpy().round(3))

## The chain

`AutoregressiveFlow` is the posterior model of the course: the product rule
$\hat q(\theta \mid x) = \hat q(x_0 \mid \psi_1)\, \hat q(y_0 \mid \psi_2)\, \hat q(E \mid \psi_3)$, every factor a
rational-quadratic spline of 2a whose parameters come from a network of
$\psi_i = (h(x), \tilde\theta_1, \dots, \tilde\theta_{i-1})$. Before the flow, the parameters go to unbounded,
standardized coordinates $\tilde\theta$, so that every sample lies inside the prior range. `fit` trains it by
maximum likelihood on minibatches, with Adam and a validation set, and keeps the state with the lowest validation
loss.

In [ ]:
torch.manual_seed(1)  # the initial weights
flow = AutoregressiveFlow(obs_dim=x.shape[1])
history = fit(flow, theta, x, epochs=20, val_fraction=2000 / len(theta), seed=1)
print(f"{flow.n_params} parameters, {history.n_train} training pairs, {history.seconds:.0f} s")
print(f"validation loss {history.best_val_loss:.3f} nats (epoch {history.best_epoch})")

fig, ax = plt.subplots(figsize=(5.5, 3.4), layout="constrained")
ax.plot(history.epoch[1:], history.train_loss[1:], color=COLORS["train"], label="training")
ax.plot(history.epoch[1:], history.val_loss[1:], color=COLORS["val"], ls="--", label="validation")
ax.set(xlabel="Epoch", ylabel=r"Loss $-\log \hat q$ [nats]")
ax.legend(loc="upper right")
plt.show()

The loss is $-\log \hat q(\theta \mid x)$ with $\theta$ in m and MeV, a density per $\mathrm{m}^2\,\mathrm{MeV}$:
it can be negative.

## The posterior of a new event

A new event, simulated now, was never seen in training. Its posterior needs no new training: the flow is evaluated
with the new observation, and `sample` runs it in the generative direction (amortized inference).

In [ ]:
event = simulate(0.5, 0.3, 2.0, config=config, seed=1001)  # a new event: vertex (0.5, 0.3) m, 2 MeV
x_o = to_pairs(event, "charge+time")[1][0]
samples = flow.sample(x_o, 2000, seed=1).numpy()  # [2000, 3]: x0, y0 [m], E [MeV]
for i, (name, unit) in enumerate([("x0", "m"), ("y0", "m"), ("E", "MeV")]):
    print(f"{name}: posterior mean {samples[:, i].mean():6.3f}, standard deviation {samples[:, i].std():.3f} {unit}")

charge = event.response.charge[0]
sx, sy = sensor_positions(config.geometry)
barycenter = np.array([charge @ sx, charge @ sy]) / charge.sum()  # the naive vertex of stage 1
R = config.geometry.radius

fig, (display, zoom, energy) = plt.subplots(1, 3, figsize=(13.0, 4.2), layout="constrained")
for ax in (display, zoom):
    phi = np.linspace(0, 2 * np.pi, 400)
    ax.plot(R * np.cos(phi), R * np.sin(phi), color="0.6", lw=1.0)
    hits = ax.scatter(sx, sy, c=charge, cmap="viridis", s=50, marker="s", zorder=3)
    ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3, color=COLORS["samples"], label=r"samples of $\hat q$")
    ax.plot(0.5, 0.3, "*", ms=14, color=COLORS["truth"], label="true vertex")
    ax.plot(*barycenter, "X", ms=10, color=COLORS["barycenter"], label="charge barycenter")
    ax.set(xlabel="$x_0$ [m]", ylabel="$y_0$ [m]", aspect="equal")
display.set(xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), title="Event display")
fig.colorbar(hits, ax=display, label="Sensor charge $n_j$", fraction=0.046, pad=0.04)
display.legend(loc="lower left", fontsize=8)
zoom.set(xlim=(0.35, 0.65), ylim=(0.15, 0.45), title="Zoom on the vertex")
energy.hist(
    samples[:, 2],
    bins=50,
    density=True,
    histtype="stepfilled",
    color=COLORS["samples"],
    alpha=0.6,
    label=r"samples of $\hat q$",
)
energy.axvline(2.0, color=COLORS["truth"], ls="--", label="true energy")
energy.set(xlabel="$E$ [MeV]", ylabel="Posterior density [1/MeV]", title="Energy")
energy.legend(loc="upper right", fontsize=8)
plt.show()

## Density and samples

The two directions of the flow: `log_prob(theta, x)` gives the density (the normalizing direction), `sample(x, n)`
the samples (the generative direction). Both take many observations at once; parameters outside the prior range get
$\log \hat q = -\infty$.

In [ ]:
truth = torch.tensor([0.5, 0.3, 2.0])
outside = torch.tensor([0.9, 0.9, 2.0])  # r0 = 1.27 m: outside the detector
print("log q^ at the truth:  ", flow.log_prob(truth, x_o).item())
print("log q^ outside the disc:", flow.log_prob(outside, x_o).item())

start = time.perf_counter()
many = flow.sample(x[-1000:], 500, seed=2)  # 500 samples for each of 1000 events
print(f"samples of 1000 events: {tuple(many.shape)} in {time.perf_counter() - start:.2f} s")

## Next steps

- The tasks are in [tasks/stage2b_npe/](../tasks/stage2b_npe/README.md): one parameter with an exact posterior (N1),
  the chain written by hand and the exact posterior of three parameters (N2), the full detector (N3), two modes (N4)
  and a flow from the library zuko (N5).
- [docs/npe.md](../docs/npe.md) has the maths, the exact references and the given figures and animations.
- `python examples/npe_quickstart.py` runs the same from the command line, and `python examples/make_npe_data.py`
  makes the data of the tasks.